# 02. Semantic Search and Indexing Pipeline with FAISS
This notebook handles the **First Stage (Stage 1)** of our search engine architecture: creating vector embeddings for Upwork job postings and building an ultra-fast **FAISS** index for semantic search.

### Pipeline Overview:
1. Loading cleaned data and centralized configuration.
2. Constructing a "smart" text corpus (placing Title + Skills at the beginning to bypass token limits).
3. Generating embeddings using `SentenceTransformer` (`all-MiniLM-L6-v2`).
4. Building and normalizing a `FAISS IndexFlatIP` index for Cosine Similarity.
5. Running an interactive test of the semantic search.

In [4]:
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import json
from pathlib import Path
import joblib


C:\Develop\Recommender-System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Path Setup & Configuration Loading
Defining the base project paths and extracting model parameters (`MODEL_NAME`, `BATCH_SIZE`) from our centralized JSON config.

In [5]:
PROJECT_ROOT = Path.cwd().parent

# Paths to configs and datasets
CONFIG_PATH = PROJECT_ROOT / "configs" / "upwork-jobs.json"
INPUT_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "upwork-jobs.csv"

# Directory where we will store embeddings and FAISS index
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

# Define exact filenames for our ML models/vectors
EMBEDDINGS_PATH = ARTIFACTS_DIR / "vacancies_embeddings.joblib"
FAISS_INDEX_PATH = ARTIFACTS_DIR / "vector_index.faiss"

# Loading config
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_NAME = config.get("MODEL_NAME", "all-MiniLM-L6-v2")
BATCH_SIZE = config.get("BATCH_SIZE", 64)
EMBEDDING_COL = config.get("EMBEDDING_COLUMN", "text_for_embedding")

## 2. Text Corpus Preparation (Feature Engineering)
**Crucial Detail:** Our base model has a context limit of 256 tokens. If we append the skills to the end of a long job description, the model will simply truncate them.
To fix this, we apply a prompt engineering hack: we construct the string in the order of **`Job Title -> Skills -> Description`**. This ensures the most critical information is guaranteed to be captured by the neural network's attention mechanism.

We also slice the dataset to a 1000-row `DEBUG` limit for rapid interactive testing.

In [7]:
# 1. Load the preprocessed dataset
df = pd.read_csv(INPUT_CSV_PATH)
print(f"Loaded {len(df)} cleaned vacancies.")

# 2. Combine Job Title and Description into a single corpus for the transformer model
# We put Job Title first so the model assigns higher semantic weight to the role name
skills_text = " ".join(df['skills'].astype(str))
df[EMBEDDING_COL] = "Job: " + df['Job Title'].astype(str) + " | Skills: " + skills_text + " | Description: " + df['Description'].astype(str)


# Inspect the combined text of the first vacancy
print("\n--- Sample text for embedding ---")
print(df['text_for_embedding'].head())

Loaded 22746 cleaned vacancies.

--- Sample text for embedding ---
0    Job: help adding html iframe within storyblok ...
1    Job: kajabi custom code | Skills: web developm...
2    Job: appointment setter | Skills: legal schedu...
3    Job: interior designer architect needed create...
4    Job: copywriter creative agency landing page |...
Name: text_for_embedding, dtype: object


In [6]:
# 1. Get the model name from the configuration
print(f"Loading transformer model: {MODEL_NAME}...")

# 2. Initialize the model
model = SentenceTransformer(MODEL_NAME)
print("Model loaded successfully!")
print(f"Max sequence length (tokens limit): {model.max_seq_length}")

Loading transformer model: nomic-ai/nomic-embed-text-v1.5...


Loading weights: 100%|██████████| 112/112 [00:00<00:00, 3088.20it/s]


Model loaded successfully!
Max sequence length (tokens limit): 8192


## 4. Vector Embeddings Generation
Encoding our textual corpus. We protect the pipeline from hidden `NaN`s (floats) by strictly forcing string types and filtering out empty rows. The resulting matrix is saved to our artifacts folder via `joblib`.

In [9]:
print(f"Starting embedding generation for all {len(df)} vacancies...")
print(f"Using batch size: {BATCH_SIZE}")

# 1. Extract the text column as a flat list
df['text_for_embedding'] = df['text_for_embedding'].fillna("").astype(str)
df_subset = df[df['text_for_embedding'].str.strip() != ""]
all_texts = df_subset['text_for_embedding'].astype(str).tolist()

# 2. Run the transformer model over the entire corpus
# show_progress_bar=True will show a nice loading bar right in DataSpell
all_embeddings = model.encode(
    all_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True
)

print("\nEncoding finished successfully!")
print(f"Final embeddings matrix shape: {all_embeddings.shape}")

# 3. Save the calculated embeddings matrix to artifacts folder
print(f"Saving embeddings to: {EMBEDDINGS_PATH}")
joblib.dump(all_embeddings, EMBEDDINGS_PATH)
print("Artifact saved")

Starting embedding generation for all 22746 vacancies...
Using batch size: 64


Batches:   0%|          | 0/356 [00:00<?, ?it/s]


Encoding finished successfully!
Final embeddings matrix shape: (22746, 768)
Saving embeddings to: /content/artifacts/vacancies_embeddings.joblib
Artifact saved


## 5. FAISS Index Creation & Normalization
We use `IndexFlatIP` (Inner Product). To make the inner product behave exactly like **Cosine Similarity**, we must explicitly normalize our embedding matrix using the `faiss.normalize_L2` function.

In [8]:
print("Initializing FAISS index...")

# 1. Get the dimensionality of the embeddings
dimension = all_embeddings.shape[1]

# 2. Create a Flat IP index.
index = faiss.IndexFlatIP(dimension)

# 3. Normalize the embeddings
faiss.normalize_L2(all_embeddings)

# 4. Add the embeddings to the index
index.add(all_embeddings)

print(f"FAISS Index successfully created! Total vectors indexed: {index.ntotal}")

# 5. Saving the index
print(f"Saving FAISS index to: {FAISS_INDEX_PATH}")
faiss.write_index(index, str(FAISS_INDEX_PATH))
print("Index artifact saved completely!")

Initializing FAISS index...


NameError: name 'all_embeddings' is not defined

## 6. System Verification: Interactive Semantic Search
Testing our system. The user's query is vectorized by the same model, undergoes L2 normalization, and is sent to FAISS to instantly retrieve the top 5 nearest neighbors.

In [7]:
test_query = "ML Developer"

print(f"Searching for: '{test_query}'\n")

# 1. Making a query vector
query_vector = model.encode([test_query])

# 2. Normalize the query vector
faiss.normalize_L2(query_vector)

# 3. Look up the nearest neighbors in the FAISS index
# k=5 means we will return the top 5 nearest neighbors
k = 5
distances, indices = index.search(query_vector, k)

# 4. Print the top 5 matching vacancies
print("Top 5 matching vacancies found:")
print("=" * 60)

for rank, (idx, score) in enumerate(zip(indices[0], distances[0]), 1):
    # Find the original row in the DataFrame
    original_row = df_subset.iloc[idx]

    print(f"Rank {rank} | Similarity Score: {score:.4f}")
    print(f"Job Title: {original_row['Job Title']}")
    print(f"Skills: {original_row['skills']}")
    print(f"Hourly Rate: ${original_row['hour_rate']:.2f}/hr")
    print(f"Snippet: {original_row['Description'][:150]}...")
    print("-" * 60)

Searching for: 'ML Developer'



NameError: name 'index' is not defined